# Análisis exploratorio y modelos predictivos de precios mayoristas en el Perú

Este cuaderno recorre el flujo del proyecto: datos, limpieza, EDA, construcción de variables y
comparación de modelos. No es un paper econométrico y no busca demostrar causalidad.

| Elemento | Valor |
|---|---|
| Fuente | SISAP–MIDAGRI, Precios de ciudades |
| Unidad | 24 departamentos × 9 categorías × mes |
| Muestra limpia | 43 855 observaciones |
| Ventana densa | 2012–2023 |
| Modelos | OLS, Ridge, Random Forest y LightGBM |
| Prueba final | enero 2021–octubre 2026 |

## Alcance y unidad de análisis

El catálogo `data/reference/departamentos.csv` contiene los **24 departamentos oficiales**, con
código ISO, código SISAP, nombre, coordenadas y zona natural. Todos los scripts leen ese catálogo.

SISAP también publica Andahuaylas, Chota, Jaén y Callao. No se incluyen: los tres primeros son
provincias de Apurímac y Cajamarca; Callao es provincia constitucional. Tratarlos como
departamentos duplicaría la unidad de análisis.

### Glosario

| Término en el texto | Nombre interno | Significado |
|---|---|---|
| Estimación del mes corriente | `nowcast` | Estimar el cambio del mes actual con información del mismo mes |
| Validación | `development` | Período 2018–2020 para elegir el modelo |
| Prueba final | `holdout` | Período 2021–octubre 2026, usado una sola vez |
| Pronóstico a un mes | `forecast` | Estimar el precio del mes siguiente |
| Referencia | `baseline` | Predicción simple contra la que se compara el modelo |

## Preguntas e hipótesis

**Preguntas**

1. ¿Cómo se distribuyen, se cubren y se relacionan los precios mayoristas de Lima y los departamentos?
2. ¿Qué modelo estima mejor el cambio del mes corriente y cuál generaliza mejor al pronosticar un mes?

**Hipótesis**

- **H1:** el cambio de precio de Lima del mismo mes contiene información útil para estimar el cambio de precio en los departamentos.
- **H2:** el desempeño predictivo es heterogéneo por producto y por horizonte; un modelo bueno para el mes corriente no necesariamente sirve para pronosticar el mes siguiente.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

RAIZ = Path.cwd()
if not (RAIZ / "data").exists():
    RAIZ = RAIZ.parent
TAB = RAIZ / "outputs" / "informe" / "tablas"
AN = RAIZ / "outputs" / "analisis"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

panel = pd.read_csv(RAIZ / "data" / "processed" / "panel_precios_limpio.csv", parse_dates=["fecha"])
print(f"{len(panel):,} filas | {panel.region.nunique()} departamentos | {panel.categoria.nunique()} productos")
print(f"{panel.fecha.min():%Y-%m} a {panel.fecha.max():%Y-%m}")

43,855 filas | 24 departamentos | 9 productos
1997-01 a 2026-11


## 1. Datos y auditoría

La fuente no es un panel balanceado: 1997–2001 cubre sobre todo Lima y la cobertura departamental se
vuelve densa en 2012. La tabla del dataset y la auditoría documentan la muestra efectiva.

In [2]:
display(pd.read_csv(TAB / "tabla_01_variables.csv"))
display(pd.read_csv(TAB / "tabla_01b_ejemplo.csv"))
display(pd.read_csv(TAB / "tabla_02_auditoria.csv"))
display(pd.read_csv(TAB / "tabla_22_verificacion_limpieza.csv"))

,variable,tipo,unidad,rol,fuente
0,fecha,fecha,mes,índice temporal,SISAP
1,anio,entero,año,derivada,fecha
2,mes,entero,1-12,derivada,fecha
3,region,texto,24 departamentos,unidad transversal,SISAP
4,categoria,texto,9 categorías,unidad de producto armonizado,SISAP
5,precio_kg,decimal,S/ por kg,variable de análisis,SISAP normalizado
6,n_registros,entero,conteo,control de agregación,derivada


,fecha,anio,mes,departamento_id,departamento,region,categoria,n_registros,precio_kg
0,2008-09-01,2008,9,PE-AMA,Amazonas,Amazonas,Aceite,1,5.56
1,2008-10-01,2008,10,PE-AMA,Amazonas,Amazonas,Aceite,1,6.39
2,2008-11-01,2008,11,PE-AMA,Amazonas,Amazonas,Aceite,1,6.72
3,2008-12-01,2008,12,PE-AMA,Amazonas,Amazonas,Aceite,1,6.64
4,2009-01-01,2009,1,PE-AMA,Amazonas,Amazonas,Aceite,1,6.34
5,2009-02-01,2009,2,PE-AMA,Amazonas,Amazonas,Aceite,1,5.71
6,2009-03-01,2009,3,PE-AMA,Amazonas,Amazonas,Aceite,1,5.54
7,2009-04-01,2009,4,PE-AMA,Amazonas,Amazonas,Aceite,1,5.66


,metrica,valor
0,filas,43921
1,departamentos,24
2,productos,9
3,nulos,0
4,duplicados fecha-departamento-producto,0
5,atipicos detectados,66
6,atipicos (%),0.15
7,primer mes,1997-01
8,ultimo mes,2026-11
9,cobertura 2012-2023 (%),85.9


,metrica,valor
0,filas antes,43921.0
1,filas despues,43855.0
2,filas descartadas,66.0
3,atipicos marcados,66.0
4,nulos antes,0.0
5,nulos despues,0.0
6,duplicados antes,0.0
7,duplicados despues,0.0
8,cobertura 2012-2023 antes (%),85.9
9,cobertura 2012-2023 despues (%),85.7


![](../outputs/informe/figuras/23_verificacion_limpieza.png)

## 2. Cobertura del panel

La figura usa las 9 categorías del panel analítico, no el conteo bruto de variedades. En 2000 hay
22 de los 24 departamentos y 7 categorías. La cobertura se calcula contra la grilla
departamento-categoría-mes.

![](../outputs/informe/figuras/02_cobertura.png)

## 3. Descripción de precios

Los precios son asimétricos a la derecha. El log reduce la asimetría en 9 de 9 productos. Por eso se
trabaja en logaritmos y se evalúa con MAE y R², no con promedios simples.

In [3]:
display(pd.read_csv(TAB / "tabla_03_descriptivos.csv").round(3))
display(pd.read_csv(TAB / "tabla_20_transformacion_log.csv").round(3))

,producto,n,media,desv,min,p25,mediana,p75,max,asimetria,curtosis
0,Papa,4074,1.235,0.508,0.370,0.846,1.120,1.532,4.670,1.077,1.719
1,Cebolla,6135,1.346,0.723,0.140,0.826,1.200,1.690,6.670,1.775,5.926
2,Zanahoria,5484,1.277,0.724,0.220,0.800,1.110,1.536,6.930,2.007,6.199
3,Azucar,4795,2.459,0.667,1.051,1.952,2.369,2.812,4.369,0.534,-0.438
4,Arroz,4651,2.686,0.567,1.209,2.303,2.607,3.050,5.400,0.398,0.076
5,Limon,4277,2.241,1.511,0.270,1.295,1.980,2.710,13.500,2.887,13.073
6,Tomate,4314,1.783,0.793,0.430,1.242,1.602,2.100,8.500,1.607,4.335
7,Aceite,3938,7.273,1.738,3.458,6.339,6.750,7.333,13.298,1.552,1.618
8,Huevos,6187,4.753,1.594,1.800,3.660,4.400,5.305,11.190,1.153,0.928


,producto,asimetria_precio,asimetria_log,curtosis_precio,curtosis_log,reduce_asimetria,reduce_curtosis
0,Papa,1.077,0.090,1.719,-0.523,True,True
1,Cebolla,1.775,-0.067,5.926,0.054,True,True
2,Zanahoria,2.007,0.107,6.199,0.234,True,True
3,Azucar,0.534,-0.009,-0.438,-0.549,True,True
4,Arroz,0.398,-0.215,0.076,0.068,True,True
5,Limon,2.887,0.005,13.073,0.464,True,True
6,Tomate,1.607,0.159,4.335,0.164,True,True
7,Aceite,1.552,1.102,1.618,0.862,True,True
8,Huevos,1.153,0.413,0.928,-0.158,True,True


![](../outputs/informe/figuras/03_campanas_precio.png)

![](../outputs/informe/figuras/04_campanas_log.png)

## 4. Correlación y geografía

La relación con Lima no es uniforme. Para papa, Ica y Huánuco están relativamente integrados;
Tumbes y Loreto casi no comparten movimiento. La distancia ayuda a explicar el gradiente, pero no
lo explica todo.

In [4]:
display(pd.read_csv(TAB / "tabla_13_geografia.csv").round(3))
display(pd.read_csv(TAB / "tabla_12_correlacion_lima.csv").head(10).round(3))

,region,r_lima,distancia_km,zona
0,Ica,0.710,265.676,Costa
1,Huanuco,0.698,251.386,Sierra
2,La Libertad,0.663,489.267,Costa
3,Arequipa,0.579,765.665,Costa
4,Lambayeque,0.521,662.548,Costa
5,Pasco,0.464,174.461,Sierra
6,Ayacucho,0.460,329.959,Sierra
7,Ancash,0.457,285.276,Sierra
8,Tacna,0.411,985.030,Costa
9,Junin,0.362,199.008,Sierra


,producto,region,n,correlacion
0,Papa,Ancash,137,0.376
1,Papa,Apurimac,123,0.484
2,Papa,Arequipa,140,0.763
3,Papa,Ayacucho,131,0.502
4,Papa,Cajamarca,131,0.307
5,Papa,Cusco,92,0.245
6,Papa,Huancavelica,65,0.330
7,Papa,Huanuco,136,0.771
8,Papa,Ica,144,0.849
9,Papa,Junin,68,0.392


![](../outputs/informe/figuras/06_correlacion_papa.png)

![](../outputs/informe/figuras/07_correlacion_lima.png)

![](../outputs/informe/figuras/08_integracion_distancia.png)

## 5. Ventanas temporales y tamaño de muestra

No hay tres estudios distintos: hay una sola fuente y tres cortes temporales.

| Ventana | Uso |
|---|---|
| 1997–2026 | Fuente completa |
| 2012–2023 | EDA y ventana densa |
| 2012–2017 | Entrenamiento |
| 2018–2020 | Validación |
| 2021–octubre 2026 | Prueba final |
| Noviembre 2026 | Fuera de la prueba |

Una observación mensual es un mes con datos completos después de construir log-precio, cambios y
rezagos. El `N` cambia por faltantes, series truncadas, productos ausentes y rezagos.

In [5]:
display(pd.read_csv(AN / "06_modelos" / "n_etapas.csv"))

,producto,tarea,entrenamiento_2012_2017,validacion_2018_2020,prueba_2021_2026,total
0,Papa,estimación del mes corriente,67,28,60,156
1,Papa,pronóstico a un mes,69,29,62,160
2,Cebolla,estimación del mes corriente,67,17,60,145
3,Cebolla,pronóstico a un mes,69,19,62,150
4,Zanahoria,estimación del mes corriente,52,17,52,121
5,Zanahoria,pronóstico a un mes,57,19,53,129
6,Arroz,estimación del mes corriente,68,27,31,127
7,Arroz,pronóstico a un mes,71,26,36,133
8,Aceite,estimación del mes corriente,68,32,24,124
9,Aceite,pronóstico a un mes,71,33,24,128


Papa y cebolla tienen más historia disponible; arroz y aceite pierden meses en la prueba final.
Cebolla tiene 17 observaciones de validación, por eso queda fuera del ranking aunque tenga 60 en la
prueba final. No es una inconsistencia.

## 6. De la tabla a la matriz X/y

Para cada producto se construye una tabla mensual de Puno y Lima. La variable objetivo es el cambio
logarítmico de Puno. Los predictores son el cambio de Lima contemporáneo y sus rezagos, los rezagos
propios y dos armónicos anuales.

In [6]:
display(pd.read_csv(TAB / "tabla_24_matriz_diseno.csv").round(3))

,fecha,producto,region,dl0,dl1,dl2,dl3,dp1,dp2,sin1,cos1,y_cambio_log,split
0,2012-06-01,Papa,Puno,0.000,-0.134,-0.012,-0.211,0.000,-0.101,0.500,-0.866,-0.019,entrenamiento
1,2012-07-01,Papa,Puno,0.108,0.000,-0.134,-0.012,-0.019,0.000,0.000,-1.000,-0.030,entrenamiento
2,2012-08-01,Papa,Puno,0.000,0.108,0.000,-0.134,-0.030,-0.019,-0.500,-0.866,0.040,entrenamiento
3,2012-09-01,Papa,Puno,-0.066,0.000,0.108,0.000,0.040,-0.030,-0.866,-0.500,-0.081,entrenamiento
4,2026-07-01,Papa,Puno,-0.160,-0.211,-0.168,-0.194,-0.164,-0.025,0.000,-1.000,-0.067,holdout
5,2026-08-01,Papa,Puno,0.419,-0.160,-0.211,-0.168,-0.067,-0.164,-0.500,-0.866,-0.038,holdout
6,2026-09-01,Papa,Puno,0.381,0.419,-0.160,-0.211,-0.038,-0.067,-0.866,-0.500,0.217,holdout
7,2026-10-01,Papa,Puno,-0.009,0.381,0.419,-0.160,0.217,-0.038,-1.000,0.000,0.159,holdout


`dl0` es el cambio de Lima del mismo mes; `dp1` y `dp2` son cambios propios rezagados; `sin1` y
`cos1` capturan estacionalidad. La tabla muestra cuatro filas del inicio y cuatro del final; el CSV
contiene la matriz completa.

## 7. Modelos

Se comparan OLS, Ridge, Random Forest y LightGBM. La selección se hace por rango promedio de R² en
validación; la prueba final se usa una sola vez. La referencia de la estimación contemporánea es
predecir cambio cero; la del pronóstico es la caminata aleatoria.

In [7]:
display(pd.read_csv(TAB / "tabla_08_torneo.csv").round(3))
display(Markdown((AN / "06_modelos" / "torneo_ganador.md").read_text(encoding="utf-8")))
holdout = pd.read_csv(TAB / "tabla_09_ganador.csv")
display(holdout[(holdout.periodo == "holdout") & (holdout.modelo == "Random Forest")][["producto", "r2", "acierto", "n"]].round(3))

,modelo,rank_promedio,r2_medio,r2_mediano,acierto_medio,productos_rankeados
0,Random Forest,1.50,-0.687,-0.097,46.879,4
1,Ridge,1.75,-0.661,-0.137,46.228,4
2,OLS,2.75,-0.689,-0.153,46.228,4
3,LightGBM,4.00,-2.434,-0.544,42.785,4


# Resultado del ranking de estimación contemporánea

**Objetivo principal:** anticipacion contemporanea del cambio mensual.

- **Modelo adoptado:** Random Forest
- Rango promedio por R² en validación: 1.50
- R² mediano en validación: -0.097
- Ranking calculado sobre 4 productos con muestras suficientes: Papa, Arroz, Aceite, Huevos
- Excluidos del ranking por no alcanzar los mínimos muestrales: Cebolla, Zanahoria
- Mínimos por producto para rankear: 60 observaciones de entrenamiento y 20 de evaluación
- Holdout: 3 de 6 productos con R2 > 0; 3 con acierto > 55 %
- Compuerta: superada

### Holdout por producto

| Producto | R2 | Acierto | n |
|---|---:|---:|---:|
| Papa | 0.270 | 69.5 % | 60 |
| Cebolla | 0.218 | 64.4 % | 60 |
| Zanahoria | 0.080 | 52.9 % | 52 |
| Arroz | -0.842 | 41.9 % | 31 |
| Aceite | -0.103 | 50.0 % | 24 |
| Huevos | -0.013 | 55.3 % | 38 |

## Justificacion

La ventana de validación es 2018-2020 (36 meses calendario) y se usa para evaluar/rankear; cada modelo se entrena con las observaciones disponibles de 2012 a 2017. Solo se rankean productos con al menos 60 observaciones de entrenamiento y 20 de evaluación. El R² es la métrica principal de la estimación contemporánea porque el MAE sobre cambios favorece predecir cero. La prueba final 2021-01 a 2026-10 se usó una sola vez; noviembre queda fuera porque no se verificó su fecha de cierre/publicacion.

¿Por qué no gana LightGBM si tiene mayor acierto en la prueba final? LightGBM acierta el signo en 62.2 % de los meses, frente a 58.0 % de Random Forest, pero el acierto es una lectura binaria: trata igual un error de 0,1 % que uno de 5 %. El ganador se eligió en validación con R², donde Random Forest tiene rango promedio 1.50 y acierto medio 46.9 %, contra 4.00 y 42.8 % de LightGBM. En la prueba final Random Forest tiene menor MAE que LightGBM en 5 de 6 productos y R² positivo en papa, cebolla y zanahoria; LightGBM tiene R² muy negativo en arroz y zanahoria. Cambiar el ganador según la prueba final sería seleccionar sobre el conjunto de prueba. La matriz de confusión se reporta como diagnóstico complementario, no como criterio de selección.

## Pronóstico a un mes: prueba final

Cambio porcentual del MAE frente a la caminata aleatoria: positivo = mejora; negativo = empeora.

| Producto | N | OLS | Ridge | Random Forest | LightGBM |
|---|---:|---:|---:|---:|---:|
| Papa | 62 | +14.5 % | +12.7 % | -74.2 % | -85.6 % |
| Cebolla | 62 | +8.3 % | +9.9 % | -2.8 % | -12.5 % |
| Zanahoria | 53 | -4.7 % | -7.7 % | -62.0 % | -59.8 % |
| Arroz | 36 | +3.9 % | +7.7 % | -282.0 % | -353.2 % |
| Aceite | 24 | -37.8 % | -36.6 % | -303.1 % | -306.8 % |
| Huevos | 38 | -3.6 % | -3.3 % | -146.4 % | -142.3 % |

Ridge y OLS mejoran en Papa, Cebolla, Arroz; ambos empeoran en Zanahoria, Aceite, Huevos. Random Forest y LightGBM empeoran frente al baseline en los seis productos. La referencia de Lima aporta valor contemporáneo, pero no ofrece un pronóstico general a un mes.


,producto,r2,acierto,n
23,Papa,0.270,69.492,60
28,Cebolla,0.218,64.407,60
33,Zanahoria,0.080,52.941,52
38,Arroz,-0.842,41.935,31
43,Aceite,-0.103,50.000,24
48,Huevos,-0.013,55.263,38


![](../outputs/informe/figuras/11_ranking_torneo.png)

![](../outputs/informe/figuras/12_ganador_vs_observado.png)

La figura compara los cuatro modelos con el cambio observado en los seis productos. No es una curva
del ganador: permite ver si un modelo suaviza demasiado, si sigue el signo o si falla en un producto.

### Pronóstico a un mes

El pronóstico se evalúa en niveles y se compara con la caminata aleatoria. La tabla muestra la
mejora porcentual del MAE; positiva significa que el modelo mejora la referencia.

In [8]:
display(pd.read_csv(TAB / "tabla_23_pronostico_holdout.csv").round(1))

,producto,n,OLS,Ridge,Random Forest,LightGBM
0,Papa,62,14.5,12.7,-74.2,-85.6
1,Cebolla,62,8.3,9.9,-2.8,-12.5
2,Zanahoria,53,-4.7,-7.7,-62.0,-59.8
3,Arroz,36,3.9,7.7,-282.0,-353.2
4,Aceite,24,-37.8,-36.6,-303.1,-306.8
5,Huevos,38,-3.6,-3.3,-146.4,-142.3


## 8. Evaluación por dirección

Además de R² y MAE, se convierte el signo del cambio en una clasificación binaria. Los meses con
cambio exactamente cero se excluyen porque no tienen dirección. La matriz de confusión muestra los
errores de signo que las métricas continuas ocultan.

In [9]:
display(pd.read_csv(TAB / "tabla_25_confusion_modelos.csv").round(1))

,modelo,TP,FN,FP,TN,n,acierto
0,LightGBM,97,53,46,66,262,62.2
1,OLS,99,51,56,56,262,59.2
2,Random Forest,92,58,52,60,262,58.0
3,Ridge,99,51,56,56,262,59.2


![](../outputs/informe/figuras/24_matriz_confusion.png)

## 9. Representación del bosque

El primer árbol del Random Forest se parte por el cambio de Lima (`dl0`), luego por el cambio propio
(`dp2`) y por estacionalidad. La importancia y la dependencia parcial muestran qué variables usa el
modelo, sin implicar causalidad.

![](../outputs/informe/figuras/15_arbol_rf.png)

![](../outputs/informe/figuras/17_importancia_rf.png)

![](../outputs/informe/figuras/18_pdp.png)

## 10. Veredicto y límites

- La relación con Lima existe, pero es heterogénea por producto y departamento.
- Random Forest es el mejor ranking de estimación contemporánea, aunque no gana en todos los productos.
- Ridge y OLS son mejores para el pronóstico a un mes en papa, cebolla y arroz.
- El pronóstico empeora en zanahoria, aceite y huevos.
- El diseño es predictivo y asociativo; no permite afirmar causalidad.
- No se verificó el rezago de publicación, así que no se habla de una estimación operativa.
- La tabla fue construida por el equipo: cada decisión de variedad, unidad y limpieza debe declararse.

El detalle metodológico está en `docs/metodologia.md`; las decisiones, en `docs/plan_desarrollo.md`.